# Solution R4 · Warm-up: the first logistic regression and the threshold

**Type**: Implement with scikit-learn

## Objective

Train a logistic regression in a `Pipeline` with scaling, read accuracy and AUC, and see what happens to **precision and recall** when the decision threshold is moved away from 0.5.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)                       # 1 = malignant
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X_tr.shape, X_te.shape, "| share of malignant cases:", round(float(y.mean()), 3))

## Commented solution

In [ ]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
test_acc = accuracy_score(y_te, pipe.predict(X_te))
test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
print(f"accuracy {test_acc:.3f} | AUC {test_auc:.3f}")

In [ ]:
p_mal = pipe.predict_proba(X_te)[:, 1]
table = pd.DataFrame(index=[0.5, 0.2, 0.05], columns=["recall", "precision"], dtype=float)
for threshold in table.index:
    predicted = (p_mal >= threshold).astype(int)
    table.loc[threshold] = [recall_score(y_te, predicted), precision_score(y_te, predicted)]
table.round(3)

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
recalls = np.array([recall_score(y_te, (p_mal >= s).astype(int)) for s in thresholds])
screening_threshold = float(thresholds[recalls >= 0.98].max())
print(f"highest threshold with recall >= 0.98: {screening_threshold:.2f}")

In [ ]:
without = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
acc_without_scaler, iter_without = without.score(X_te, y_te), int(without.n_iter_[0])
print(f"without scaler: accuracy {acc_without_scaler:.3f}, iterations {iter_without} (with the scaler: {int(pipe[-1].n_iter_[0])})")

## Check

In [ ]:
with verify():
    check(0.93 < test_acc < 0.99 and test_auc > 0.98, "accuracy between 0.93 and 0.99, AUC over 0.98", "auc = roc_auc_score(y_te, p_mal)")
    check(list(table.index) == [0.5, 0.2, 0.05] and table["recall"].is_monotonic_increasing and table["precision"].is_monotonic_decreasing, "lowering the threshold the recall rises and the precision drops", "recall_score and precision_score with (p_mal >= threshold)")
    check(table.loc[0.05, "recall"] >= 0.98 and table.loc[0.05, "precision"] < table.loc[0.5, "precision"] - 0.05, "with threshold 0.05 the recall reaches 0.98 or more, at the expense of precision (over 5 points less)", "compare the rows of the table")
    check(0.02 < screening_threshold < 0.4 and recall_score(y_te, (p_mal >= screening_threshold).astype(int)) >= 0.98, "the screening threshold is low and gives a recall of at least 0.98", "thresholds[recalls >= 0.98].max()")
    check(acc_without_scaler < test_acc and iter_without >= 1000, "without scaler the accuracy is lower and the solver hits the limit of 1000 iterations without converging", "without.score(X_te, y_te), without.n_iter_[0]")
done()

## What you learned

- **Accuracy and AUC say different things:** with threshold 0.5 the model has accuracy 0.986 and AUC 0.996. The AUC measures how well the scores **rank** the cases, independently of the threshold.
- **The threshold is a decision, not a parameter of the model:** lowering it from 0.5 to 0.05 raises the recall of the malignant cases from 0.96 to 1.00 (fewer missed tumors) and lowers the precision from 1.00 to 0.78 (more false alarms). Which trade-off to choose depends on the cost of the two errors, not on the statistics: it is the theme of exercises B2 and D2 of this module.
- **Scaling is not a detail here either:** without `StandardScaler` the logistic regression does not converge within 1000 iterations (with the scaler 30 are enough) and the accuracy drops from 0.986 to 0.965. It is the subject of lesson 4.2.

## Common pitfall

Reporting only the accuracy at threshold 0.5 as if it were "the model": the threshold is a decision choice, and with imbalanced classes or different costs 0.5 is almost never the right one.

## Interview question

<details><summary>Why does lowering the threshold increase the recall and reduce the precision?</summary>

Lowering the threshold classifies more cases as positive: more of the true positives are caught (recall up) but more false positives come in too (precision down). The threshold is chosen with the costs of the two errors and validated on data not used to choose it.

</details>